# Entry Evaluation v2 — Legacy Gold (Damerau-Levenshtein)

This notebook reimplements the P1-style entry-level evaluation using legacy
gold annotations from two annotation rounds (Tahap 1, Tahap 2). It complements
`Gold_Annotation_Comparison.ipynb` (which evaluates the 5 new-gold dicts at
row level with IAA-validated parcor). Together the two notebooks cover all
twelve dicts that have human-validated gold data.

## Scope

- **Dicts evaluated**: 18, 34, 42, 54, 68, 71, 89 (legacy gold available)
- **Resources evaluated**: bilingual lexicon, morphology, parallel corpus
- **Methodology**: P1 (Daniel et al., 2023) — entry-level Damerau-Levenshtein
  normalized similarity per component
- **Gold annotation method**: single-keyed (no IAA — that's why the new-gold
  comparison was built separately for parcor rigor)

## Three evaluation stages

**Stage 1 — Source word recall.** For each legacy gold lemma, does the
pipeline contain that lemma anywhere (in the bilingual lexicon or morphology
output for that dict)? Reported as the fraction of gold lemmas the pipeline
successfully extracted.

**Stage 2 — Component placement accuracy.** For lemmas the pipeline did
extract, score each of `kata_tujuan` (target word), `form` (POS), and `makna`
(meaning) against the gold annotation using Damerau-Levenshtein normalized
similarity (from `textdistance`). Reported per component, separately for
bilingual lexicon (`kata_tujuan`, `makna`) and morphology (`form`). `makna`
is sparse in gold (~14% filled); rows with empty gold makna are skipped.

**Stage 3 — Parallel corpus sentence accuracy.** For legacy gold rows that
contain a parcor pair, score the pipeline's parcor for that lemma against the
gold using DL similarity on each side (`kalimat_asal`, `kalimat_tujuan`).
Reported as the fraction of gold pairs found in the pipeline plus the average
similarity score.

## Outputs (in `../csvAnalysis/legacy_evaluation/`)

- `<dict_id>_legacy_eval.csv` — per-gold-row scores
- `<dict_id>_legacy_summary.csv` — per-dict aggregate metrics
- `_combined_summary.csv` — cross-dict comparison

## 1. Configuration

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import textdistance
import re

# === Legacy dicts (have gold annotation in either tahap) ===
LEGACY_DICTS = ["18", "34", "42", "54", "68", "71", "89"]

# === Legacy gold files ===
GOLD_DIR = Path("../csvAnalysis/legacy_gold_diagnostic/")
GOLD_TAHAP1 = GOLD_DIR / "GoldEntries_Tahap1_Merged.xlsx"
GOLD_TAHAP2 = GOLD_DIR / "GoldEntries_Tahap2_Merged.xlsx"

# === Pipeline output paths ===
BILLEX_DIR = Path("../Ekstraksi/9. Bilingual Lexicon - Fixed")
MORPH_DIR  = Path("../Ekstraksi/10. Morphology - Fixed")
PARCOR_DIR = Path("../Ekstraksi/11. Parallel Corpus - Fixed")

# Some pipelines store morphology directly in the Ekstraksi root; check both
MORPH_FALLBACK_DIR = Path("../Ekstraksi")

# === Similarity thresholds for "match" classification ===
# DL normalized similarity in [0, 1]. P1 reports buckets at 0.80, 0.85, 0.90, 1.00.
DL_BUCKETS = [1.00, 0.95, 0.90, 0.85, 0.80]
DL_MATCH_THRESHOLD = 0.85  # for binary match/no-match classification

# === Output ===
OUT_DIR = Path("../csvAnalysis/legacy_evaluation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Legacy dicts: {LEGACY_DICTS}")
print(f"Gold files: {GOLD_TAHAP1}, {GOLD_TAHAP2}")
print(f"Output dir: {OUT_DIR.resolve()}")

Legacy dicts: ['18', '34', '42', '54', '68', '71', '89']
Gold files: ..\csvAnalysis\legacy_gold_diagnostic\GoldEntries_Tahap1_Merged.xlsx, ..\csvAnalysis\legacy_gold_diagnostic\GoldEntries_Tahap2_Merged.xlsx
Output dir: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\csvAnalysis\legacy_evaluation


## 2. Load and consolidate legacy gold from both tahaps

The two tahap files store annotations as Excel workbooks, one sheet per dict.
We concatenate Tahap 1 and Tahap 2 for each dict — they have zero lemma
overlap (verified separately), so concatenation is safe.

In [6]:
def load_tahap(path, tahap_label):
    """Load all sheets from an annotation xlsx and tag with dict + tahap."""
    if not path.exists():
        print(f"  ⚠ Gold file missing: {path}")
        return pd.DataFrame()
    xl = pd.ExcelFile(path)
    frames = []
    for sheet in xl.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        df["dict_id"] = sheet
        df["tahap"] = tahap_label
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


gold_t1 = load_tahap(GOLD_TAHAP1, "1")
gold_t2 = load_tahap(GOLD_TAHAP2, "2")
gold_all = pd.concat([gold_t1, gold_t2], ignore_index=True)

# Normalize string columns
for col in ["kata_asal", "kata_dasar", "kata_tujuan", "makna", "form",
            "kalimat_asal", "kalimat_tujuan", "annotator", "source_file"]:
    if col in gold_all.columns:
        gold_all[col] = gold_all[col].fillna("").astype(str).str.strip()

print(f"Tahap 1: {len(gold_t1)} rows")
print(f"Tahap 2: {len(gold_t2)} rows")
print(f"Combined: {len(gold_all)} rows")
print(f"\nPer-dict breakdown:")
print(gold_all.groupby(["dict_id", "tahap"]).size().to_string())

Tahap 1: 172 rows
Tahap 2: 849 rows
Combined: 1021 rows

Per-dict breakdown:
dict_id  tahap
18       2        260
34       1         26
42       1         40
54       1         29
         2        184
68       2        178
71       1         40
         2        109
89       1         37
         2        118


## 3. Parse list-formatted fields

The legacy gold stores `kata_tujuan`, `makna` as string representations of
lists like `'[mengeram]'`, `'[]'`, or `'[a, b]'`. We parse these into actual
Python lists for matching. `kata_dasar` uses `'-'` as a placeholder for empty.

In [7]:
def parse_list_field(value: str) -> list:
    """Parse a string like '[a, b, c]' or '[mengeram]' into a list of strings.

    Returns empty list for '[]', '-', '', or any unparseable input.
    """
    if not value or value in ("[]", "-", "nan", "NaN"):
        return []
    v = value.strip()
    # Strip enclosing brackets if present
    if v.startswith("[") and v.endswith("]"):
        v = v[1:-1].strip()
    if not v:
        return []
    # Split on commas (handles 'a, b, c' inside brackets)
    parts = [p.strip().strip("'\"") for p in v.split(",")]
    return [p for p in parts if p]


def normalize_field(value: str) -> str:
    """For plain-string fields like `form` and `kata_asal`. '-' → empty."""
    if not value or value in ("-", "nan", "NaN"):
        return ""
    return value.strip()


# Apply parsers
gold_all["kata_asal_norm"] = gold_all["kata_asal"].apply(normalize_field)
gold_all["kata_dasar_norm"] = gold_all["kata_dasar"].apply(normalize_field)
gold_all["form_norm"] = gold_all["form"].apply(normalize_field)
gold_all["kata_tujuan_list"] = gold_all["kata_tujuan"].apply(parse_list_field)
gold_all["makna_list"] = gold_all["makna"].apply(parse_list_field)

# Sanity check
print("=== Field fill rates after normalization ===")
for col, label in [
    ("kata_asal_norm",   "kata_asal (headword)"),
    ("kata_dasar_norm",  "kata_dasar (base form)"),
    ("form_norm",        "form (POS)"),
    ("kata_tujuan_list", "kata_tujuan (target word list)"),
    ("makna_list",       "makna (meaning list)"),
    ("kalimat_asal",     "kalimat_asal (source sentence)"),
    ("kalimat_tujuan",   "kalimat_tujuan (target sentence)"),
]:
    if col.endswith("_list"):
        n_filled = (gold_all[col].apply(len) > 0).sum()
    else:
        n_filled = (gold_all[col] != "").sum()
    print(f"  {label}: {n_filled} / {len(gold_all)} = {n_filled/len(gold_all):.1%}")

=== Field fill rates after normalization ===
  kata_asal (headword): 1021 / 1021 = 100.0%
  kata_dasar (base form): 357 / 1021 = 35.0%
  form (POS): 808 / 1021 = 79.1%
  kata_tujuan (target word list): 863 / 1021 = 84.5%
  makna (meaning list): 144 / 1021 = 14.1%
  kalimat_asal (source sentence): 620 / 1021 = 60.7%
  kalimat_tujuan (target sentence): 603 / 1021 = 59.1%


## 4. DL similarity helpers

Wraps `textdistance.damerau_levenshtein.normalized_similarity`. Two variants:

- **`dl_sim(a, b)`** — scalar string comparison. Returns 1.0 for two empty
  strings, 0.0 if exactly one is empty.
- **`dl_sim_list_vs_list(gold_list, pipe_list)`** — symmetric list-to-list
  similarity. For each gold item, find its best DL match in the pipeline list;
  for each pipeline item, find its best DL match in the gold list; average all
  the maxima. Symmetric — penalizes both missing translations and spurious
  extra ones.
- **`dl_sim_list_best(gold_list, pipeline_value)`** — convenience wrapper that
  parses `pipeline_value` as a stringified list (since the pipeline schema
  stores `kata_tujuan` as `"['kakak', 'abang']"`) and delegates to
  `dl_sim_list_vs_list`.

The list-to-list comparison is needed because both gold and pipeline store
multi-item fields (`kata_tujuan`, `makna`) as Python list literals in CSV
strings. Comparing them as raw strings systematically depresses similarity
because of bracket/quote/comma noise even when the actual content is
identical.

In [8]:
def dl_sim(a: str, b: str) -> float:
    """Damerau-Levenshtein normalized similarity in [0, 1]."""
    a = (a or "").strip()
    b = (b or "").strip()
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return textdistance.damerau_levenshtein.normalized_similarity(a, b)


def dl_sim_list_vs_list(gold_list: list, pipe_list: list) -> float:
    """Symmetric list-to-list similarity using best-bipartite matching.

    For each gold item, find its best DL match in the pipeline list.
    For each pipeline item, find its best DL match in the gold list.
    Return the average of these maxima (symmetric, penalizes both missing
    and spurious items).

    Returns NaN if gold is empty (no expectation). Returns 0.0 if gold is
    non-empty but pipeline is empty.
    """
    if not gold_list:
        return float("nan")  # caller should skip
    if not pipe_list:
        return 0.0

    # Gold-side: for each gold item, best match in pipeline
    gold_best = [max(dl_sim(g, p) for p in pipe_list) for g in gold_list]
    # Pipeline-side: for each pipeline item, best match in gold
    pipe_best = [max(dl_sim(p, g) for g in gold_list) for p in pipe_list]
    # Symmetric: average across all maxima
    all_max = gold_best + pipe_best
    return sum(all_max) / len(all_max)


def dl_sim_list_best(gold_list: list, pipeline_value) -> float:
    """Backward-compat wrapper. Takes pipeline_value as either a string (legacy)
    or a list-formatted string (current pipeline schema) and parses accordingly.
    """
    if not gold_list:
        return float("nan")
    pipe_list = parse_list_field(pipeline_value) if pipeline_value else []
    return dl_sim_list_vs_list(gold_list, pipe_list)


# Smoke tests
print(f"DL('hello', 'hello'): {dl_sim('hello', 'hello')}")
print(f"DL('hello', 'helo'): {dl_sim('hello', 'helo'):.4f}")
print(f"DL('hello', 'world'): {dl_sim('hello', 'world'):.4f}")
print()
print("List comparison tests:")
print(f"  Identical lists ['a','b'] vs ['a','b']: {dl_sim_list_vs_list(['a','b'], ['a','b']):.4f}")
print(f"  Gold ['kakak','abang'] vs pipe ['kakak','abang']: {dl_sim_list_vs_list(['kakak','abang'], ['kakak','abang']):.4f}")
print(f"  Gold ['kakak','abang'] vs pipe ['kakak']: {dl_sim_list_vs_list(['kakak','abang'], ['kakak']):.4f}")
print(f"  Gold ['kakak'] vs pipe ['kakak','spurious']: {dl_sim_list_vs_list(['kakak'], ['kakak','spurious']):.4f}")
print()
print("Pipeline-as-stringified-list (the dict 89 case):")
raw_pipeline = "['kakak', 'abang']"
print(f"  Gold ['kakak','abang'] vs pipe \"{raw_pipeline}\": {dl_sim_list_best(['kakak','abang'], raw_pipeline):.4f}")

DL('hello', 'hello'): 1.0
DL('hello', 'helo'): 0.8000
DL('hello', 'world'): 0.2000

List comparison tests:
  Identical lists ['a','b'] vs ['a','b']: 1.0000
  Gold ['kakak','abang'] vs pipe ['kakak','abang']: 1.0000
  Gold ['kakak','abang'] vs pipe ['kakak']: 0.7333
  Gold ['kakak'] vs pipe ['kakak','spurious']: 0.6667

Pipeline-as-stringified-list (the dict 89 case):
  Gold ['kakak','abang'] vs pipe "['kakak', 'abang']": 1.0000


## 5. Load pipeline outputs per dict

For each legacy dict, load the three pipeline output files. Some dicts may
have missing files (e.g., morphology empty); those are handled gracefully.

In [9]:
def load_pipeline_outputs(dict_id: str) -> dict:
    """Returns dict with keys 'billex', 'morph', 'parcor'. Missing files become empty DataFrames."""
    result = {}

    # Bilingual lexicon
    billex_path = BILLEX_DIR / f"{dict_id}_Billex.csv"
    if billex_path.exists():
        df = pd.read_csv(billex_path)
        for c in df.columns:
            if df[c].dtype == object:
                df[c] = df[c].fillna("").astype(str).str.strip()
        result["billex"] = df
    else:
        result["billex"] = pd.DataFrame()

    # Morphology — try main dir, then fallback
    for mp in [MORPH_DIR / f"{dict_id}_Morphology.csv",
               MORPH_FALLBACK_DIR / f"{dict_id}_Morphology.csv"]:
        if mp.exists():
            df = pd.read_csv(mp)
            for c in df.columns:
                if df[c].dtype == object:
                    df[c] = df[c].fillna("").astype(str).str.strip()
            result["morph"] = df
            break
    else:
        result["morph"] = pd.DataFrame()

    # Parcor (audit version preferred)
    parcor_path = PARCOR_DIR / f"{dict_id}_Parcor_audit.csv"
    if parcor_path.exists():
        df = pd.read_csv(parcor_path)
        for c in df.columns:
            if df[c].dtype == object:
                df[c] = df[c].fillna("").astype(str).str.strip()
        result["parcor"] = df
    else:
        result["parcor"] = pd.DataFrame()

    return result


# Preview load status for each dict
print("=== Pipeline file availability per dict ===")
pipeline_per_dict = {}
for did in LEGACY_DICTS:
    out = load_pipeline_outputs(did)
    pipeline_per_dict[did] = out
    print(f"  Dict #{did}:  billex={len(out['billex'])}  morph={len(out['morph'])}  parcor={len(out['parcor'])}")

=== Pipeline file availability per dict ===
  Dict #18:  billex=9877  morph=2684  parcor=2341
  Dict #34:  billex=5730  morph=2322  parcor=4407
  Dict #42:  billex=3225  morph=1037  parcor=3683
  Dict #54:  billex=1085  morph=728  parcor=1174
  Dict #68:  billex=6936  morph=2343  parcor=2044
  Dict #71:  billex=783  morph=28  parcor=727
  Dict #89:  billex=1128  morph=106  parcor=935


## 6. Stage 1 — Source word recall

For each legacy gold lemma, search the pipeline's bilingual lexicon and
morphology for that lemma. We check multiple potential column names because
naming varies across pipeline iterations.

In [10]:
def find_lemma_in_pipeline(lemma: str, pipeline: dict) -> dict:
    """Returns dict with 'in_billex', 'in_morph', 'billex_row', 'morph_row'.

    The billex_row and morph_row are the matched DataFrame rows for use in Stage 2.
    """
    result = {"in_billex": False, "in_morph": False, "billex_row": None, "morph_row": None}
    if not lemma:
        return result

    # Bilingual lexicon: try 'kata_asal', 'lemma', 'headword', 'kata'
    billex = pipeline["billex"]
    if len(billex):
        for col in ["kata_asal", "lemma", "headword", "kata"]:
            if col in billex.columns:
                matches = billex[billex[col].astype(str).str.strip() == lemma]
                if len(matches):
                    result["in_billex"] = True
                    result["billex_row"] = matches.iloc[0]
                    break

    # Morphology: usually 'kata' (main lemma) or 'form' (derived form)
    morph = pipeline["morph"]
    if len(morph):
        for col in ["kata", "form", "lemma", "kata_asal"]:
            if col in morph.columns:
                matches = morph[morph[col].astype(str).str.strip() == lemma]
                if len(matches):
                    result["in_morph"] = True
                    result["morph_row"] = matches.iloc[0]
                    break

    return result


# Quick check on dict 18
sample_pipe = pipeline_per_dict.get("18", {"billex": pd.DataFrame(), "morph": pd.DataFrame()})
sample_gold = gold_all[gold_all["dict_id"] == "18"].head(5)
print("=== Sample Stage 1 lookups (dict 18) ===")
for _, row in sample_gold.iterrows():
    res = find_lemma_in_pipeline(row["kata_asal_norm"], sample_pipe)
    print(f"  '{row['kata_asal_norm']}': billex={res['in_billex']}, morph={res['in_morph']}")

=== Sample Stage 1 lookups (dict 18) ===
  'angkrem': billex=False, morph=False
  'angkrem': billex=False, morph=False
  'angkrik': billex=False, morph=False
  'angkrik-angkrik': billex=True, morph=False
  'angkring': billex=True, morph=False


## 7. Stage 2 — Component placement accuracy

For lemmas found in the pipeline, score each component against the gold value
using DL normalized similarity. Components scored separately:

- **`kata_tujuan`** (bilingual lexicon): gold is a list; we take the best DL
  similarity across list items against the pipeline's single value
- **`form`** (morphology): gold and pipeline are both plain strings
- **`makna`** (bilingual lexicon): gold is a list (sparse — ~14% filled); we
  skip rows where gold makna is empty

In [11]:
def score_component(gold_row, pipeline_match) -> dict:
    """For a single gold-row + pipeline-match pair, score each component.

    Returns dict with similarity scores per component. Score is NaN when the
    component cannot be evaluated (e.g., gold value missing, pipeline column
    not present).
    """
    scores = {
        "kata_tujuan_dl": float("nan"),
        "form_dl":        float("nan"),
        "makna_dl":       float("nan"),
    }

    # === kata_tujuan from bilingual lexicon ===
    if pipeline_match["in_billex"]:
        gold_kt_list = gold_row["kata_tujuan_list"]
        pipe_kt = ""
        billex_row = pipeline_match["billex_row"]
        # Try common column names for the target word
        for col in ["kata_tujuan", "target", "translation"]:
            if billex_row is not None and col in billex_row.index:
                pipe_kt = str(billex_row[col]).strip()
                if pipe_kt:
                    break
        if gold_kt_list:  # only score when gold has expectation
            scores["kata_tujuan_dl"] = dl_sim_list_best(gold_kt_list, pipe_kt)

    # === form (POS) from morphology ===
    if pipeline_match["in_morph"]:
        gold_form = gold_row["form_norm"]
        pipe_form = ""
        morph_row = pipeline_match["morph_row"]
        for col in ["tag", "form", "pos"]:
            if morph_row is not None and col in morph_row.index:
                pipe_form = str(morph_row[col]).strip()
                if pipe_form:
                    break
        if gold_form:
            scores["form_dl"] = dl_sim(gold_form, pipe_form)

    # === makna from bilingual lexicon ===
    if pipeline_match["in_billex"]:
        gold_makna_list = gold_row["makna_list"]
        pipe_makna = ""
        billex_row = pipeline_match["billex_row"]
        for col in ["makna", "meaning", "definition"]:
            if billex_row is not None and col in billex_row.index:
                pipe_makna = str(billex_row[col]).strip()
                if pipe_makna:
                    break
        if gold_makna_list:  # skip rows where gold makna is empty
            scores["makna_dl"] = dl_sim_list_best(gold_makna_list, pipe_makna)

    return scores


print("Stage 2 scorer defined.")

Stage 2 scorer defined.


## 8. Stage 3 — Parallel corpus sentence accuracy

For legacy gold rows that contain both `kalimat_asal` and `kalimat_tujuan`,
look up the pipeline's parcor row for that lemma and score the two sides
separately with DL similarity.

The pipeline's audit file may have multiple rows per lemma (sub-entries are
separate rows). We take the **best-matching** row across all pipeline rows
sharing that lemma — i.e., for each gold pair, pick the pipeline pair with
highest combined DL similarity.

In [12]:
def score_parcor(gold_row, pipeline: dict) -> dict:
    """Score the gold parcor pair against the pipeline's parcor for the same lemma."""
    scores = {
        "parcor_asal_dl":     float("nan"),
        "parcor_tujuan_dl":   float("nan"),
        "parcor_avg_dl":      float("nan"),
        "parcor_found":       False,
    }

    gold_asal = gold_row["kalimat_asal"]
    gold_tuj  = gold_row["kalimat_tujuan"]
    if not gold_asal or not gold_tuj:
        return scores  # gold has no parcor expectation

    lemma = gold_row["kata_asal_norm"]
    parcor = pipeline["parcor"]
    if not lemma or len(parcor) == 0:
        return scores

    # Find pipeline rows matching this lemma
    matches = None
    for col in ["lemma", "kata_asal", "main_lemma"]:
        if col in parcor.columns:
            m = parcor[parcor[col].astype(str).str.strip() == lemma]
            if len(m):
                matches = m
                break
    if matches is None or len(matches) == 0:
        return scores  # parcor doesn't have this lemma

    # Identify the asal/tuj columns in pipeline parcor
    asal_col = next((c for c in ["kalimat_asal", "kalimat_asal_original"]
                     if c in parcor.columns), None)
    tuj_col  = next((c for c in ["kalimat_tujuan", "kalimat_tujuan_original"]
                     if c in parcor.columns), None)
    if not asal_col or not tuj_col:
        return scores

    # Score each candidate pipeline row, take the best by combined similarity
    best = None
    for _, prow in matches.iterrows():
        p_asal = str(prow[asal_col]).strip()
        p_tuj  = str(prow[tuj_col]).strip()
        if not p_asal and not p_tuj:
            continue
        s_asal = dl_sim(gold_asal, p_asal)
        s_tuj  = dl_sim(gold_tuj, p_tuj)
        avg = (s_asal + s_tuj) / 2
        if best is None or avg > best["avg"]:
            best = {"asal": s_asal, "tuj": s_tuj, "avg": avg}

    if best:
        scores["parcor_found"] = True
        scores["parcor_asal_dl"]   = best["asal"]
        scores["parcor_tujuan_dl"] = best["tuj"]
        scores["parcor_avg_dl"]    = best["avg"]

    return scores


print("Stage 3 scorer defined.")

Stage 3 scorer defined.


## 9. Run evaluation per dict — produce per-row scored file

In [13]:
def evaluate_dict(dict_id: str) -> pd.DataFrame:
    """Run all three stages on every gold row for this dict."""
    gold = gold_all[gold_all["dict_id"] == dict_id].copy().reset_index(drop=True)
    pipeline = pipeline_per_dict[dict_id]

    rows = []
    for _, g in gold.iterrows():
        lemma = g["kata_asal_norm"]
        # Stage 1
        match = find_lemma_in_pipeline(lemma, pipeline)
        # Stage 2
        s2 = score_component(g, match)
        # Stage 3
        s3 = score_parcor(g, pipeline)

        rows.append({
            "dict_id":            dict_id,
            "tahap":              g["tahap"],
            "annotator":          g.get("annotator", ""),
            "kata_asal":          lemma,
            "in_billex":          match["in_billex"],
            "in_morph":           match["in_morph"],
            "found_anywhere":     match["in_billex"] or match["in_morph"],
            "gold_kata_tujuan":   "|".join(g["kata_tujuan_list"]),
            "gold_form":          g["form_norm"],
            "gold_makna":         "|".join(g["makna_list"]),
            "gold_kalimat_asal":  g["kalimat_asal"],
            "gold_kalimat_tujuan":g["kalimat_tujuan"],
            "kata_tujuan_dl":     s2["kata_tujuan_dl"],
            "form_dl":            s2["form_dl"],
            "makna_dl":           s2["makna_dl"],
            "parcor_found":       s3["parcor_found"],
            "parcor_asal_dl":     s3["parcor_asal_dl"],
            "parcor_tujuan_dl":   s3["parcor_tujuan_dl"],
            "parcor_avg_dl":      s3["parcor_avg_dl"],
        })
    return pd.DataFrame(rows)


all_eval = {}
for did in LEGACY_DICTS:
    df = evaluate_dict(did)
    all_eval[did] = df
    eval_path = OUT_DIR / f"{did}_legacy_eval.csv"
    df.to_csv(eval_path, index=False)
    n_rows = len(df)
    n_found = df["found_anywhere"].sum()
    print(f"  Dict #{did}: {n_rows} gold rows, {n_found} found ({n_found/n_rows:.1%} recall)")

print(f"\nPer-row score files written to {OUT_DIR}")

  Dict #18: 260 gold rows, 172 found (66.2% recall)
  Dict #34: 26 gold rows, 4 found (15.4% recall)
  Dict #42: 40 gold rows, 8 found (20.0% recall)
  Dict #54: 213 gold rows, 61 found (28.6% recall)
  Dict #68: 178 gold rows, 158 found (88.8% recall)
  Dict #71: 149 gold rows, 105 found (70.5% recall)
  Dict #89: 155 gold rows, 148 found (95.5% recall)

Per-row score files written to ..\csvAnalysis\legacy_evaluation


## 10. Per-dict aggregate summaries

Compute the headline metrics for each dict:

- **Stage 1**: source word recall (% of gold lemmas the pipeline found)
- **Stage 2**: component placement — mean DL similarity per component over
  rows where the component could be scored (i.e., gold value present AND
  pipeline found the lemma). Also report fraction above thresholds 0.85, 0.90, 1.00.
- **Stage 3**: parcor sentence accuracy — fraction of gold parcor pairs found,
  mean DL similarity per side, fraction above 0.85.

In [14]:
def summarize_dict(did: str, df: pd.DataFrame) -> dict:
    summary = {"dict_id": did}
    n = len(df)
    summary["n_gold_rows"] = n
    if n == 0:
        return summary

    # Stage 1
    summary["recall_anywhere"] = round(df["found_anywhere"].mean(), 4)
    summary["recall_billex"]   = round(df["in_billex"].mean(), 4)
    summary["recall_morph"]    = round(df["in_morph"].mean(), 4)

    # Stage 2 — for each component, compute over rows where score is not NaN
    for comp in ["kata_tujuan_dl", "form_dl", "makna_dl"]:
        scored = df[df[comp].notna()]
        n_scored = len(scored)
        summary[f"{comp}_n"] = n_scored
        if n_scored:
            summary[f"{comp}_mean"]    = round(scored[comp].mean(), 4)
            summary[f"{comp}_pct_085"] = round((scored[comp] >= 0.85).mean(), 4)
            summary[f"{comp}_pct_090"] = round((scored[comp] >= 0.90).mean(), 4)
            summary[f"{comp}_pct_100"] = round((scored[comp] >= 1.00).mean(), 4)
        else:
            summary[f"{comp}_mean"]    = float("nan")
            summary[f"{comp}_pct_085"] = float("nan")
            summary[f"{comp}_pct_090"] = float("nan")
            summary[f"{comp}_pct_100"] = float("nan")

    # Stage 3
    parcor_gold = df[(df["gold_kalimat_asal"] != "") & (df["gold_kalimat_tujuan"] != "")]
    n_parcor_gold = len(parcor_gold)
    summary["n_parcor_gold"] = n_parcor_gold
    if n_parcor_gold:
        summary["parcor_found_rate"]  = round(parcor_gold["parcor_found"].mean(), 4)
        found = parcor_gold[parcor_gold["parcor_found"]]
        if len(found):
            summary["parcor_asal_dl_mean"]   = round(found["parcor_asal_dl"].mean(), 4)
            summary["parcor_tujuan_dl_mean"] = round(found["parcor_tujuan_dl"].mean(), 4)
            summary["parcor_avg_dl_mean"]    = round(found["parcor_avg_dl"].mean(), 4)
            summary["parcor_pct_avg_085"]    = round((found["parcor_avg_dl"] >= 0.85).mean(), 4)
        else:
            for c in ["parcor_asal_dl_mean", "parcor_tujuan_dl_mean",
                      "parcor_avg_dl_mean", "parcor_pct_avg_085"]:
                summary[c] = float("nan")
    else:
        for c in ["parcor_found_rate", "parcor_asal_dl_mean",
                  "parcor_tujuan_dl_mean", "parcor_avg_dl_mean", "parcor_pct_avg_085"]:
            summary[c] = float("nan")

    return summary


summary_rows = [summarize_dict(did, all_eval[did]) for did in LEGACY_DICTS]
summary_df = pd.DataFrame(summary_rows)
combined_path = OUT_DIR / "_combined_summary.csv"
summary_df.to_csv(combined_path, index=False)

print("=== Per-dict aggregate summary ===")
# Print only the headline columns
headline_cols = [
    "dict_id", "n_gold_rows",
    "recall_anywhere", "recall_billex", "recall_morph",
    "kata_tujuan_dl_mean", "kata_tujuan_dl_n",
    "form_dl_mean", "form_dl_n",
    "makna_dl_mean", "makna_dl_n",
    "n_parcor_gold", "parcor_found_rate", "parcor_avg_dl_mean",
]
existing = [c for c in headline_cols if c in summary_df.columns]
print(summary_df[existing].to_string(index=False))
print(f"\nWritten: {combined_path.name}")

=== Per-dict aggregate summary ===
dict_id  n_gold_rows  recall_anywhere  recall_billex  recall_morph  kata_tujuan_dl_mean  kata_tujuan_dl_n  form_dl_mean  form_dl_n  makna_dl_mean  makna_dl_n  n_parcor_gold  parcor_found_rate  parcor_avg_dl_mean
     18          260           0.6615         0.6462        0.2269               0.8219               128        0.7799         53         0.3878          29              0                NaN                 NaN
     34           26           0.1538         0.1538        0.1538               0.2273                 4        0.0000          2         0.0541           1              1             0.0000                 NaN
     42           40           0.2000         0.2000        0.1250               0.0324                 3        1.0000          4         0.2801           4             22             0.3182              0.0959
     54          213           0.2864         0.2817        0.1127               0.9722                53        0.44

## 11. Per-dict summary CSV files

In [15]:
for did in LEGACY_DICTS:
    s_row = next((s for s in summary_rows if s["dict_id"] == did), None)
    if s_row is None:
        continue
    sdf = pd.DataFrame([s_row])
    spath = OUT_DIR / f"{did}_legacy_summary.csv"
    sdf.to_csv(spath, index=False)

print(f"Per-dict summary files written.")
print(f"All outputs in: {OUT_DIR}")
for p in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {p.name}")

Per-dict summary files written.
All outputs in: ..\csvAnalysis\legacy_evaluation
  18_legacy_eval.csv
  18_legacy_summary.csv
  34_legacy_eval.csv
  34_legacy_summary.csv
  42_legacy_eval.csv
  42_legacy_summary.csv
  54_legacy_eval.csv
  54_legacy_summary.csv
  68_legacy_eval.csv
  68_legacy_summary.csv
  71_legacy_eval.csv
  71_legacy_summary.csv
  89_legacy_eval.csv
  89_legacy_summary.csv
  _combined_summary.csv


## 12. Final cross-dict readout

In [16]:
print("=" * 70)
print("  Legacy Gold Entry-Level Evaluation — Summary")
print("=" * 70)
print(f"\n  Dicts evaluated:        {len(LEGACY_DICTS)} ({', '.join(LEGACY_DICTS)})")
print(f"  Total legacy gold rows: {len(gold_all)}")
print(f"  Methodology:            Damerau-Levenshtein normalized similarity (textdistance)")

# Headline numbers across all dicts
print(f"\n  === STAGE 1: SOURCE WORD RECALL ===")
for did in LEGACY_DICTS:
    s = next(s for s in summary_rows if s["dict_id"] == did)
    print(f"    Dict #{did}: {s.get('recall_anywhere', 0):.1%} (n={s.get('n_gold_rows', 0)})")

print(f"\n  === STAGE 2: COMPONENT PLACEMENT (mean DL similarity) ===")
print(f"    {'Dict':<8} {'kata_tujuan':<15} {'form':<15} {'makna':<15}")
for did in LEGACY_DICTS:
    s = next(s for s in summary_rows if s["dict_id"] == did)
    kt = s.get("kata_tujuan_dl_mean", float("nan"))
    fm = s.get("form_dl_mean", float("nan"))
    mk = s.get("makna_dl_mean", float("nan"))
    kt_s = f"{kt:.3f} (n={s.get('kata_tujuan_dl_n', 0)})" if not pd.isna(kt) else "n/a"
    fm_s = f"{fm:.3f} (n={s.get('form_dl_n', 0)})" if not pd.isna(fm) else "n/a"
    mk_s = f"{mk:.3f} (n={s.get('makna_dl_n', 0)})" if not pd.isna(mk) else "n/a"
    print(f"    #{did:<7} {kt_s:<15} {fm_s:<15} {mk_s:<15}")

print(f"\n  === STAGE 3: PARALLEL CORPUS (mean DL similarity, asal+tuj avg) ===")
for did in LEGACY_DICTS:
    s = next(s for s in summary_rows if s["dict_id"] == did)
    n = s.get("n_parcor_gold", 0)
    fr = s.get("parcor_found_rate", float("nan"))
    avg = s.get("parcor_avg_dl_mean", float("nan"))
    if not pd.isna(fr) and not pd.isna(avg):
        print(f"    Dict #{did}: found={fr:.1%} (n={n}), mean DL when found={avg:.3f}")
    else:
        print(f"    Dict #{did}: n={n} parcor gold pairs")

print(f"\n  This evaluation complements Gold_Annotation_Comparison.ipynb")
print(f"  which evaluates new-gold dicts 4, 19, 24, 46, 91 at row level.")

  Legacy Gold Entry-Level Evaluation — Summary

  Dicts evaluated:        7 (18, 34, 42, 54, 68, 71, 89)
  Total legacy gold rows: 1021
  Methodology:            Damerau-Levenshtein normalized similarity (textdistance)

  === STAGE 1: SOURCE WORD RECALL ===
    Dict #18: 66.1% (n=260)
    Dict #34: 15.4% (n=26)
    Dict #42: 20.0% (n=40)
    Dict #54: 28.6% (n=213)
    Dict #68: 88.8% (n=178)
    Dict #71: 70.5% (n=149)
    Dict #89: 95.5% (n=155)

  === STAGE 2: COMPONENT PLACEMENT (mean DL similarity) ===
    Dict     kata_tujuan     form            makna          
    #18      0.822 (n=128)   0.780 (n=53)    0.388 (n=29)   
    #34      0.227 (n=4)     0.000 (n=2)     0.054 (n=1)    
    #42      0.032 (n=3)     1.000 (n=4)     0.280 (n=4)    
    #54      0.972 (n=53)    0.449 (n=23)    n/a            
    #68      0.698 (n=105)   0.938 (n=87)    0.576 (n=54)   
    #71      0.968 (n=103)   0.667 (n=4)     0.250 (n=4)    
    #89      0.890 (n=145)   0.800 (n=20)    0.375 (n=8)    